In [1]:
import pandas as pd

data = [
    {"customer_id":1001, "name":"Asha Mehta",  "phone":"+91-9876543210", "address":"23 Lodi Rd, Delhi",     "plan_type":"Prepaid",  "arpu_inr":180},
    {"customer_id":1002, "name":"Ravi Kumar",  "phone":"+91-9988776655", "address":"12 Marine Dr, Mumbai",  "plan_type":"Postpaid", "arpu_inr":280},
    {"customer_id":1003, "name":"Sneha Rao",   "phone":"+91-9090909090", "address":"7 Anna Salai, Chennai", "plan_type":"Prepaid",  "arpu_inr":210},
    {"customer_id":1004, "name":"Manoj Singh", "phone":"+91-9123456789", "address":"44 CR Park, Delhi",     "plan_type":"Postpaid", "arpu_inr":320},
    {"customer_id":1005, "name":"Divya Jain",  "phone":"+91-9000011111", "address":"18 Park St, Kolkata",   "plan_type":"Prepaid",  "arpu_inr":120}
]

pd.DataFrame(data).to_csv("customer_data.csv", index=False)

print("✅ customer_data.csv created")


✅ customer_data.csv created


In [2]:
import pandas as pd
from datetime import datetime

AUDIT_LOG = "access_audit.log"

# Audit Log Function
def log_access(role, fields_returned):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open(AUDIT_LOG, "a", encoding="utf-8") as f:
        f.write(f"[{ts}] role={role} fields={fields_returned}\n")


# Role Policies
POLICY = {
    "junior_analyst": {
        "name": "hide",
        "phone": "hide",
        "address": "hide",
        "plan_type": "show",
        "arpu_inr": "show"
    },
    "senior_analyst": {
        "name": "hide",
        "phone": "mask",
        "address": "hide",
        "plan_type": "show",
        "arpu_inr": "show"
    },
    "manager": {
        "name": "show",
        "phone": "show",
        "address": "show",
        "plan_type": "show",
        "arpu_inr": "show"
    }
}


# Apply Masking
def apply_masking(df, role):
    if role not in POLICY:
        raise ValueError(f"Unknown role: {role}")

    pol = POLICY[role]
    out = df.copy()

    # Name
    if pol.get("name") == "hide" and "name" in out.columns:
        out["name"] = "ANONYMIZED"

    # Phone
    if "phone" in out.columns:
        if pol.get("phone") == "hide":
            out["phone"] = "XXXXX"
        elif pol.get("phone") == "mask":
            out["phone"] = out["phone"].astype(str).str[:3] + "-XXX-XXXX"

    # Address
    if pol.get("address") == "hide" and "address" in out.columns:
        out["address"] = "HIDDEN"

    return out


# Role-Based Access Function
def get_customer_data(role, fields=None):
    df = pd.read_csv("customer_data.csv")

    mandatory = ["customer_id"]

    if fields is None:
        fields = list(df.columns)

    fields = list(dict.fromkeys(mandatory + fields))
    df = df[fields]

    masked = apply_masking(df, role)
    log_access(role, fields)

    return masked

In [3]:
junior = get_customer_data("junior_analyst")

print("Junior Analyst View:")
print(junior)

junior.to_csv("view_junior.csv", index=False)

print("✅ view_junior.csv created")

Junior Analyst View:
   customer_id        name  phone address plan_type  arpu_inr
0         1001  ANONYMIZED  XXXXX  HIDDEN   Prepaid       180
1         1002  ANONYMIZED  XXXXX  HIDDEN  Postpaid       280
2         1003  ANONYMIZED  XXXXX  HIDDEN   Prepaid       210
3         1004  ANONYMIZED  XXXXX  HIDDEN  Postpaid       320
4         1005  ANONYMIZED  XXXXX  HIDDEN   Prepaid       120
✅ view_junior.csv created


In [4]:
senior = get_customer_data("senior_analyst")

print("Senior Analyst View:")
print(senior)

senior.to_csv("view_senior.csv", index=False)

print("✅ view_senior.csv created")

Senior Analyst View:
   customer_id        name         phone address plan_type  arpu_inr
0         1001  ANONYMIZED  +91-XXX-XXXX  HIDDEN   Prepaid       180
1         1002  ANONYMIZED  +91-XXX-XXXX  HIDDEN  Postpaid       280
2         1003  ANONYMIZED  +91-XXX-XXXX  HIDDEN   Prepaid       210
3         1004  ANONYMIZED  +91-XXX-XXXX  HIDDEN  Postpaid       320
4         1005  ANONYMIZED  +91-XXX-XXXX  HIDDEN   Prepaid       120
✅ view_senior.csv created


In [5]:
manager = get_customer_data("manager")

print("Manager View:")
print(manager)

manager.to_csv("view_manager.csv", index=False)

print("✅ view_manager.csv created")

Manager View:
   customer_id         name           phone                address plan_type  \
0         1001   Asha Mehta  +91-9876543210      23 Lodi Rd, Delhi   Prepaid   
1         1002   Ravi Kumar  +91-9988776655   12 Marine Dr, Mumbai  Postpaid   
2         1003    Sneha Rao  +91-9090909090  7 Anna Salai, Chennai   Prepaid   
3         1004  Manoj Singh  +91-9123456789      44 CR Park, Delhi  Postpaid   
4         1005   Divya Jain  +91-9000011111    18 Park St, Kolkata   Prepaid   

   arpu_inr  
0       180  
1       280  
2       210  
3       320  
4       120  
✅ view_manager.csv created


In [6]:
with open("access_audit.log", "r", encoding="utf-8") as f:
    print("Access Audit Log:\n")
    print(f.read())

Access Audit Log:

[2026-09-14 02:58:42] role=junior_analyst fields=['customer_id', 'name', 'phone', 'address', 'plan_type', 'arpu_inr']
[2026-09-14 02:59:11] role=senior_analyst fields=['customer_id', 'name', 'phone', 'address', 'plan_type', 'arpu_inr']
[2026-09-14 02:59:40] role=manager fields=['customer_id', 'name', 'phone', 'address', 'plan_type', 'arpu_inr']

